# 01 データ取得 & シグナル分析

## 目的
- J-Quants APIからデータ取得（全銘柄日足10年）
- 出来高ベースの機関投資家シグナルを作成
- 「出来高急増 = 機関投資家の資金流入」仮説を統計的に検証

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
%matplotlib inline

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')

## STEP 1: データ取得

J-Quants API Standard plan (10年ヒストリカル) からデータを取得。
Parquetキャッシュあり — 2回目以降は数秒で起動。

In [ ]:
from quant_research.data_fetcher import fetch_all, prepare_price_dataframe

# 全データ取得（初回はAPI呼び出し、2回目以降はキャッシュ）
raw_data = fetch_all(years=10)

print(f"銘柄マスター: {len(raw_data['master']):,} stocks")
print(f"株価データ:   {len(raw_data['prices']):,} rows")
print(f"TOPIX:       {len(raw_data['topix']):,} rows")
print(f"投資部門別:   {len(raw_data['investor_types']):,} rows")

In [ ]:
# 分析用DataFrame構築
df = prepare_price_dataframe(raw_data['prices'], raw_data['master'])

print(f"分析対象: {df['Code'].nunique():,} 銘柄")
print(f"期間: {df['Date'].min().date()} ~ {df['Date'].max().date()}")
print(f"行数: {len(df):,}")
df.head()

## STEP 2: 特徴量エンジニアリング

In [ ]:
from quant_research.feature_engine import compute_all_features

df = compute_all_features(df, fins=raw_data.get('fins_summary'), forward_periods=[3, 5, 10])

print(f"\n特徴量数: {len(df.columns)}")
print(f"\n特徴量一覧:")
for col in sorted(df.columns):
    print(f"  {col}")

## 出来高シグナルの分布分析

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 出来高倍率の分布
vr = df['vol_ratio'].dropna()
vr_clipped = vr.clip(upper=10)
axes[0,0].hist(vr_clipped, bins=100, color='steelblue', alpha=0.7)
axes[0,0].axvline(2.0, color='red', linestyle='--', label='2x')
axes[0,0].axvline(3.0, color='orange', linestyle='--', label='3x')
axes[0,0].set_title('出来高倍率の分布 (capped at 10x)')
axes[0,0].set_xlabel('Volume Ratio')
axes[0,0].legend()

# 出来高Zスコアの分布
vz = df['vol_zscore'].dropna().clip(-5, 10)
axes[0,1].hist(vz, bins=100, color='coral', alpha=0.7)
axes[0,1].axvline(2.0, color='red', linestyle='--', label='Z=2')
axes[0,1].set_title('出来高Zスコアの分布')
axes[0,1].set_xlabel('Volume Z-Score')
axes[0,1].legend()

# 出来高倍率と5日後リターンの関係
sample = df.dropna(subset=['vol_ratio', 'fwd_5d_return']).sample(min(50000, len(df)))
axes[1,0].scatter(sample['vol_ratio'].clip(upper=10), 
                  sample['fwd_5d_return'].clip(-0.2, 0.2) * 100,
                  alpha=0.05, s=1)
axes[1,0].set_xlabel('Volume Ratio')
axes[1,0].set_ylabel('5-Day Forward Return (%)')
axes[1,0].set_title('出来高倍率 vs 5日後リターン')
axes[1,0].axhline(0, color='gray', linestyle='-', alpha=0.3)

# 出来高倍率別の平均5日後リターン
bins = [0, 1, 1.5, 2, 2.5, 3, 4, 5, 7, 10, 100]
labels = ['<1', '1-1.5', '1.5-2', '2-2.5', '2.5-3', '3-4', '4-5', '5-7', '7-10', '10+']
df['vol_bin'] = pd.cut(df['vol_ratio'], bins=bins, labels=labels)
vol_returns = df.groupby('vol_bin', observed=False)['fwd_5d_return'].agg(['mean', 'count']).dropna()
colors = ['gray' if v < 0 else 'steelblue' for v in vol_returns['mean']]
axes[1,1].bar(vol_returns.index.astype(str), vol_returns['mean'] * 100, color=colors, alpha=0.8)
axes[1,1].set_title('出来高倍率別の平均5日後リターン (%)')
axes[1,1].set_xlabel('Volume Ratio Bin')
axes[1,1].set_ylabel('Avg 5d Return (%)')
for i, (idx, row) in enumerate(vol_returns.iterrows()):
    axes[1,1].annotate(f'n={int(row["count"]):,}', 
                       (i, row['mean']*100), 
                       textcoords='offset points', xytext=(0, 5),
                       fontsize=7, ha='center')

plt.tight_layout()
plt.savefig('../quant_research/data/reports/volume_signal_analysis.png', dpi=150)
plt.show()

## 仮説検証: 出来高急増 = 機関投資家の資金流入？

投資部門別売買状況データ（Standard plan）を使い、
出来高急増日と機関投資家の売買フローの相関を直接検証。

In [ ]:
investor_df = raw_data.get('investor_types')

if investor_df is not None and not investor_df.empty:
    print(f"投資部門別データ: {len(investor_df):,} rows")
    print(f"カラム: {list(investor_df.columns)}")
    
    from quant_research.reporter import plot_volume_institutional_correlation
    fig = plot_volume_institutional_correlation(
        df, investor_df,
        save_path='../quant_research/data/reports/vol_institutional_correlation.png'
    )
    if fig:
        plt.show()
    
    # 統計的検定
    vol_by_date = df.groupby('Date')['vol_ratio'].mean()
    high_vol_dates = vol_by_date[vol_by_date > 2.0].index
    normal_dates = vol_by_date[vol_by_date <= 2.0].index
    
    print(f"\n出来高急増日: {len(high_vol_dates):,} days")
    print(f"通常日: {len(normal_dates):,} days")
else:
    print("投資部門別データが利用できません")
    print("出来高急増後のリターンから間接的に検証します")
    
    # 間接検証: 出来高急増後のリターン分析
    high_vol = df[df['vol_ratio'] > 2.0]['fwd_5d_return'].dropna()
    normal_vol = df[df['vol_ratio'] <= 2.0]['fwd_5d_return'].dropna()
    
    t_stat, p_value = stats.ttest_ind(high_vol, normal_vol)
    print(f"\n--- 出来高急増 vs 通常: 5日後リターンの比較 ---")
    print(f"出来高急増 (>2x): 平均={high_vol.mean()*100:.3f}%, 中央値={high_vol.median()*100:.3f}%, n={len(high_vol):,}")
    print(f"通常:             平均={normal_vol.mean()*100:.3f}%, 中央値={normal_vol.median()*100:.3f}%, n={len(normal_vol):,}")
    print(f"t検定: t={t_stat:.4f}, p={p_value:.6f}")
    print(f"統計的有意差: {'あり (p<0.05)' if p_value < 0.05 else 'なし (p>=0.05)'}")

## 出来高急増時の勝率分析

In [ ]:
# 出来高倍率別の勝率テーブル
results = []
for period in [3, 5, 10]:
    ret_col = f'fwd_{period}d_return'
    for vr_min in [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
        subset = df[df['vol_ratio'] >= vr_min][ret_col].dropna()
        if len(subset) < 10:
            continue
        results.append({
            'holding': f'{period}d',
            'vol_ratio_min': f'{vr_min}x',
            'n_trades': len(subset),
            'win_rate': f"{(subset > 0).mean():.1%}",
            'avg_return': f"{subset.mean()*100:.3f}%",
            'median_return': f"{subset.median()*100:.3f}%",
        })

pd.DataFrame(results)

In [ ]:
# 中間データ保存
df.to_pickle('../quant_research/data/_intermediate_df_features.pkl')
print(f"保存完了: {len(df):,} rows, {len(df.columns)} columns")